In [ ]:
import os
os.environ["V5G0_STAGE"] = "baseline"

In [ ]:
#!/usr/bin/env python3
"""
v5 G1(a)+(b) — VCC oracle composite + per-perturbation difficulty link.
KAGGLE, CPU-only. Engineer, 2026-07-19, per Brian's ruling of 2026-07-19.

WHAT THIS ANSWERS
  (a) The day-one "reproducibility ceiling" question, now feasible: score a
      split of VCC's REAL cells against itself with `cell-eval`, Arc's own
      scorer, UNMODIFIED (locked invariant #1 — never write our own scorer).
      This is the actual number the whole v5 program has been circling.
      Brian's pre-registered forecast, made before this data existed:
        DES_scaled ~ 0.60, PDS_scaled ~ 0.975, MAE_scaled ~ 0.40
        -> composite 63.6 central, range 45.7-79.0
      Report the measured number against this forecast either way -- if it's
      off by twenty points, that goes in the record next to the forecast.
  (b) Does our direct reproducibility r_p (v5g0_05_vcc_direct_specific_v1.py)
      predict per-perturbation leaderboard difficulty? Correlate r_p against
      cell-eval's own per-perturbation oracle DES/PDS/MAE.

WHY --ceiling AND NOT OUR OWN SPLIT
  `cell-eval` (>=0.7, see https://github.com/ArcInstitute/cell-eval) ships a
  `--ceiling` mode built for exactly this: per perturbation (and control),
  cells are bootstrapped to 2x count and split into two equal halves; one
  half plays "real", the other "prediction"; the FULL metric suite (DES via
  Wilcoxon+BH-FDR, PDS via pseudobulk L1 ranking, MAE) runs on that self-split.
  That is the organizers' own definition of a reproducibility ceiling. We use
  it rather than reimplementing a split, per the locked invariant: "the moment
  we write our own scorer, the finding becomes a dispute about our code."

WHY A FRESH PASS AND NOT THE G0 ACCUMULATOR
  DES requires per-cell Wilcoxon tests; v5g0_vcc_accum_v1.npz holds pseudobulk
  sums only (built for the correlation-based G0 test, which never needed
  cell-level distributions). Brian: "don't try to force it from what's on
  disk." This script re-reads adata_Training.h5ad from scratch.

STAGES (run with --stage; default "all" runs them in order on Kaggle)
  diagnose  - confirm obs/var schema and the cell_eval API surface BEFORE
              spending compute. Fails loudly and prints exactly what it found
              if any assumption below is wrong.
  baseline  - build the naive ("predict no perturbation effect") baseline
              anndata and run cell_eval.MetricsEvaluator against real data.
              This is the scaling denominator, matching how the public
              leaderboard is normalized (implied DES_base=0.1058,
              PDS_base=0.5140, reconstructed from six leaderboard rows in
              STATE.md 2026-07-18 -- this run gives us cell-eval's own
              baseline numbers directly, a check on that reconstruction).
  oracle    - run evaluator.compute_ceiling() on the real data. This IS part
              (a): the reproducibility ceiling.
  score     - cell_eval.score_agg_metrics(oracle agg, baseline agg) -> the
              SCALED oracle composite, comparable to the 0-100ish leaderboard.
  link      - part (b): correlate per-perturbation oracle metrics against our
              r_p from v5g0_vcc_direct_specific_v1.csv.

ASSUMPTIONS FLAGGED FOR THE diagnose STAGE TO CONFIRM OR KILL
  - obs['target_gene'] is the perturbation column, obs['target_gene']==
    'non-targeting' is control (confirmed already by the G0 probe/extraction
    logs; re-checked here on the same file for a live sanity check).
  - MetricsEvaluator(adata_pred=..., adata_real=..., control_pert=...,
    pert_col=..., num_threads=...).compute_ceiling(seed=0) returns
    (ceiling_df, ceiling_agg_df) and ceiling mode does not require adata_pred
    to be a real prediction (the README says it is computed from real data
    only). We pass adata_pred=adata_real to satisfy the constructor and print
    a warning + the function's docstring if this looks wrong.
"""
import argparse
import json
import os
import subprocess
import sys
import time

import numpy as np
import pandas as pd

t0 = time.time()
def log(m): print(f"[{time.time()-t0:7.1f}s] {m}", flush=True)


# ------------------------------------------------------------- bootstrap ---
def ensure_deps():
    """cell-eval (and its anndata/pdex/etc. deps) is not preinstalled on
    Kaggle's base image. First run hit ModuleNotFoundError: No module named
    'anndata'. Installing it fixed that but exposed a second, sharper problem:
    cell-eval pulls in a newer numpy/scipy than the ones Kaggle's base kernel
    already imported at startup (pandas, etc. import numpy transitively before
    any of our code runs). pip installing a new numpy INTO a running process
    that has already imported the old one does not take effect -- Python
    cannot hot-swap a loaded C extension. The result was not an install
    failure; it was numpy's own internals partially upgraded on disk while
    the in-memory module stayed old, producing the exact confusing crash we
    hit: "cannot import name '_center' from numpy._core.umath".

    Fix: check package versions via importlib.metadata (reads installed
    package metadata from disk, does NOT import the package, so this check
    itself can't trigger the crash). If numpy or scipy changed version as a
    side effect of installing cell-eval, STOP CLEANLY and tell the user to
    restart the kernel -- do not attempt to import anything, because that
    import is exactly what will crash. This is a one-time environment fix,
    not a bug in the analysis; the second run (post-restart) sees the correct
    versions already on disk and proceeds straight through.
    """
    import importlib.metadata as im

    def ver(pkg):
        try:
            return im.version(pkg)
        except im.PackageNotFoundError:
            return None

    needed = {"anndata": "anndata", "cell_eval": "cell-eval"}
    watch = ["numpy", "scipy"]   # packages whose live C extensions we can't hot-swap
    before = {w: ver(w) for w in watch}
    missing = [pkg for mod, pkg in needed.items() if ver(mod.replace("_", "-")) is None
              and ver(mod) is None]

    if not missing:
        log("dependencies already present: " + ", ".join(needed))
    else:
        log(f"installing missing dependencies: {missing}")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
        log("pip install finished.")

    after = {w: ver(w) for w in watch}
    changed = {w: (before[w], after[w]) for w in watch if before[w] != after[w]}
    if changed:
        log("!" * 70)
        log("numpy/scipy were upgraded on disk by installing cell-eval:")
        for pkg, (b, a) in changed.items():
            log(f"    {pkg}: {b} -> {a}")
        log("This kernel already had the old versions loaded in memory before this "
            "script ran (Kaggle's base image imports numpy/pandas at startup). "
            "Python cannot hot-swap a loaded compiled extension -- importing "
            "anndata/cell_eval in THIS process will crash with a confusing "
            "ImportError (e.g. \"cannot import name '_center' from numpy._core.umath\").")
        log("ACTION REQUIRED: restart the kernel now (Kaggle: Run > Restart session, "
            "or the restart button), then re-run this cell. The correct versions are "
            "already installed on disk -- the restart just lets Python load them fresh. "
            "This is a one-time fix; the next run will not hit this branch.")
        log("!" * 70)
        sys.exit(0)

    # Safe to import now -- either nothing changed, or (post-restart) the
    # versions on disk already match what's about to be imported.
    for mod in needed:
        __import__(mod)
    log("dependencies verified importable.")


# --------------------------------------------------------------- paths -----
_ON_KAGGLE = os.path.isdir("/kaggle/input")
H5AD = os.environ.get(
    "VCC_H5AD",
    "/kaggle/input/datasets/ericdu847/v5g0-vcc-00-probe/adata_Training.h5ad"
    if _ON_KAGGLE else
    "/sessions/peaceful-cool-bohr/mnt/Project 13/data/raw/vcc/adata_Training.h5ad")
# our r_p table from v5g0_05_vcc_direct_specific_v1.py -- upload as a Kaggle
# Dataset alongside the h5ad, or override with V5G0_DIRECT_SPECIFIC_CSV.
DIRECT_SPECIFIC_CSV = os.environ.get(
    "V5G0_DIRECT_SPECIFIC_CSV",
    "/kaggle/input/datasets/ericdu847/v5g0-vcc-accum/v5g0_vcc_direct_specific_v1.csv"
    if _ON_KAGGLE else
    "/sessions/peaceful-cool-bohr/mnt/Project 13/results/v5 G0 - VCC/v5g0_vcc_direct_specific_v1.csv")
OUT = os.environ.get("V5G0_OUT", "/kaggle/working" if _ON_KAGGLE else
                     "/sessions/peaceful-cool-bohr/mnt/Project 13/results/v5 G1 - oracle")
os.makedirs(OUT, exist_ok=True)

CONTROL_LABEL_CANDIDATES = ("non-targeting", "non_targeting", "control", "NTC")
NUM_THREADS = int(os.environ.get("V5G0_THREADS", "4"))

# Brian's pre-registered forecast (2026-07-19), locked before this measurement:
FORECAST = dict(DES_scaled=0.60, PDS_scaled=0.975, MAE_scaled=0.40,
               composite_central=63.6, composite_range=(45.7, 79.0))


def find_pert_and_control(adata):
    for col in ("target_gene", "gene", "perturbation", "gene_symbol"):
        if col in adata.obs.columns:
            vals = adata.obs[col].astype(str)
            for c in CONTROL_LABEL_CANDIDATES:
                if (vals == c).any():
                    return col, c
    raise KeyError(
        f"Could not find a (perturbation column, control label) pair. "
        f"obs columns: {list(adata.obs.columns)}. "
        f"Tried control labels: {CONTROL_LABEL_CANDIDATES}. Fix explicitly, "
        f"do not guess.")


# ============================================================== diagnose ===
def stage_diagnose():
    import anndata as ad
    log(f"reading {H5AD}")
    adata = ad.read_h5ad(H5AD)
    log(f"shape: {adata.shape}")
    log(f"obs columns: {list(adata.obs.columns)}")
    log(f"var columns: {list(adata.var.columns)}, index sample: {list(adata.var.index[:5])}")

    pert_col, ctrl_label = find_pert_and_control(adata)
    log(f"perturbation column = '{pert_col}', control label = '{ctrl_label}'")
    vc = adata.obs[pert_col].value_counts()
    log(f"n perturbations (incl. control) = {vc.shape[0]}; control n_cells = {vc.get(ctrl_label)}")

    import importlib.metadata as im
    try:
        from cell_eval import MetricsEvaluator, score_agg_metrics
        cell_eval_version = im.version("cell-eval")   # __version__ attr doesn't exist on the package
        log(f"cell_eval version: {cell_eval_version}")
        log(f"MetricsEvaluator.compute_ceiling docstring:\n{MetricsEvaluator.compute_ceiling.__doc__}")
        import inspect
        log(f"MetricsEvaluator.__init__ signature: {inspect.signature(MetricsEvaluator.__init__)}")
        log(f"MetricsEvaluator.compute_ceiling signature: "
            f"{inspect.signature(MetricsEvaluator.compute_ceiling)}")
        log(f"score_agg_metrics signature: {inspect.signature(score_agg_metrics)}")
        # CONFIRMED (2026-07-19 diagnose run): adata_pred is a required positional
        # arg with no default, but compute_ceiling's docstring states plainly
        # "Uses the real data only" -- adata_pred=adata_real in stage_oracle() is
        # correct, not a guess anymore. compute()/compute_ceiling() return
        # POLARS DataFrames (not pandas) -- fixed in stage_baseline/stage_oracle
        # to use .write_csv() instead of pandas' .to_csv(path, index=False).
    except ImportError:
        log("cell_eval not installed. Run: pip install -U cell-eval")
        sys.exit(1)

    with open(os.path.join(OUT, "v5g1_diagnose_v1.json"), "w") as f:
        json.dump(dict(shape=list(adata.shape), obs_columns=list(adata.obs.columns),
                       pert_col=pert_col, ctrl_label=ctrl_label,
                       n_perturbations=int(vc.shape[0]),
                       cell_eval_version=cell_eval_version),
                 f, indent=2)
    log("diagnose complete. READ THE SIGNATURES ABOVE before running other stages -- "
        "if compute_ceiling requires something other than adata_pred=adata_real, "
        "fix stage_oracle() before spending the CPU-hours.")


# =============================================================== baseline ==
def stage_baseline():
    """Naive baseline: predict 'no perturbation effect' for every cell, i.e.
    every cell's predicted profile = the control group's mean profile. This
    is the standard naive baseline (matches 'Naive', rank 6 on the public
    leaderboard per STATE.md 2026-07-18) and is what oracle/ceiling numbers
    get scaled against, exactly as the real leaderboard does."""
    import anndata as ad
    from cell_eval import MetricsEvaluator

    log(f"reading {H5AD}")
    adata = ad.read_h5ad(H5AD)
    pert_col, ctrl_label = find_pert_and_control(adata)

    X = adata.X
    import scipy.sparse as sp
    ctrl_mask = (adata.obs[pert_col].astype(str) == ctrl_label).values
    ctrl_mean = np.asarray(X[ctrl_mask].mean(axis=0)).ravel() if sp.issparse(X) \
        else X[ctrl_mask].mean(axis=0)
    log(f"control mean profile computed from {int(ctrl_mask.sum())} cells")

    pred = adata.copy()
    n = pred.n_obs
    pred.X = np.tile(ctrl_mean, (n, 1)).astype(np.float32)
    log(f"naive baseline anndata built: every cell = control mean, shape {pred.shape}")

    evaluator = MetricsEvaluator(
        adata_pred=pred, adata_real=adata,
        control_pert=ctrl_label, pert_col=pert_col, num_threads=NUM_THREADS)
    results, agg_results = evaluator.compute()
    # compute()/compute_ceiling() return POLARS DataFrames (confirmed via the
    # diagnose stage's signature dump), not pandas -- .write_csv(), not
    # .to_csv(path, index=False); str(df) for logging, not .to_string().
    results.write_csv(os.path.join(OUT, "v5g1_baseline_results_v1.csv"))
    agg_results.write_csv(os.path.join(OUT, "v5g1_baseline_agg_results_v1.csv"))
    log(f"baseline agg_results:\n{agg_results}")
    log("baseline stage complete.")


# ================================================================= oracle ==
def stage_oracle():
    """Part (a): the reproducibility ceiling. cell-eval's own --ceiling
    mechanism: real cells bootstrapped 2x and split, full metric suite run
    self-vs-self. This is the number Brian pre-registered a forecast for."""
    import anndata as ad
    from cell_eval import MetricsEvaluator

    log(f"reading {H5AD}")
    adata = ad.read_h5ad(H5AD)
    pert_col, ctrl_label = find_pert_and_control(adata)

    # CONFIRMED (2026-07-19 diagnose run, not a guess): adata_pred is a
    # required positional arg with no default, but compute_ceiling's own
    # docstring states "Uses the real data only" -- adata_pred=adata_real is
    # correct; ceiling mode does not touch adata_pred's contents.
    evaluator = MetricsEvaluator(
        adata_pred=adata, adata_real=adata,
        control_pert=ctrl_label, pert_col=pert_col, num_threads=NUM_THREADS)
    log("running compute_ceiling(seed=0) -- this is the expensive step")
    ceiling, ceiling_agg = evaluator.compute_ceiling(seed=0)
    # polars DataFrames -- .write_csv(), not pandas' .to_csv(path, index=False)
    ceiling.write_csv(os.path.join(OUT, "v5g1_oracle_ceiling_results_v1.csv"))
    ceiling_agg.write_csv(os.path.join(OUT, "v5g1_oracle_ceiling_agg_v1.csv"))
    log(f"oracle ceiling agg_results:\n{ceiling_agg}")
    log("oracle stage complete.")


# ================================================================== score ==
def stage_score():
    """Scale the oracle ceiling against the naive baseline -> comparable to
    the public leaderboard's 0-100ish composite. Reports against Brian's
    pre-registered forecast either way."""
    from cell_eval import score_agg_metrics

    user_csv = os.path.join(OUT, "v5g1_oracle_ceiling_agg_v1.csv")
    base_csv = os.path.join(OUT, "v5g1_baseline_agg_results_v1.csv")
    for p in (user_csv, base_csv):
        if not os.path.exists(p):
            raise FileNotFoundError(f"{p} missing -- run stage_oracle/stage_baseline first.")

    out_csv = os.path.join(OUT, "v5g1_oracle_score_v1.csv")
    score_agg_metrics(results_user=user_csv, results_base=base_csv, output=out_csv)
    score_df = pd.read_csv(out_csv)
    log(f"score.csv columns: {list(score_df.columns)}")
    log(f"score.csv:\n{score_df.to_string()}")

    # Best-effort composite extraction: look for an existing overall/composite
    # column first (authoritative if cell-eval computes one); fall back to
    # the STATE.md-reconstructed weighting ONLY as a labeled cross-check.
    report = {"score_csv_columns": list(score_df.columns),
             "score_csv_raw": score_df.to_dict(orient="records"),
             "brians_preregistered_forecast": FORECAST}
    overall_col = next((c for c in score_df.columns
                        if c.lower() in ("overall", "composite", "score")), None)
    if overall_col:
        report["measured_composite_from_cell_eval"] = float(score_df[overall_col].iloc[0])
        log(f"MEASURED oracle composite (from cell-eval's own column "
            f"'{overall_col}'): {report['measured_composite_from_cell_eval']:.2f}  "
            f"vs forecast central {FORECAST['composite_central']} "
            f"(range {FORECAST['composite_range']})")
    else:
        log("No overall/composite column found in score.csv -- report DES_scaled/"
            "PDS_scaled/MAE_scaled individually against the forecast. Do NOT "
            "invent a weighting scheme to force a single number; that would be "
            "exactly the 'our own scorer' problem the locked invariant forbids.")

    with open(os.path.join(OUT, "v5g1_oracle_score_report_v1.json"), "w") as f:
        json.dump(report, f, indent=2)
    log("score stage complete.")


# =================================================================== link ==
def stage_link():
    """Part (b): does r_p (v5g0_05) predict per-perturbation oracle difficulty?"""
    ceiling_csv = os.path.join(OUT, "v5g1_oracle_ceiling_results_v1.csv")
    if not os.path.exists(ceiling_csv):
        raise FileNotFoundError(f"{ceiling_csv} missing -- run stage_oracle first.")
    ceiling = pd.read_csv(ceiling_csv)
    log(f"ceiling per-perturbation columns: {list(ceiling.columns)}")

    if not os.path.exists(DIRECT_SPECIFIC_CSV):
        raise FileNotFoundError(
            f"{DIRECT_SPECIFIC_CSV} missing -- upload v5g0_vcc_direct_specific_v1.csv "
            f"as a Kaggle Dataset input, or set V5G0_DIRECT_SPECIFIC_CSV.")
    rp = pd.read_csv(DIRECT_SPECIFIC_CSV)
    rp_real = rp[rp.run_kind == "real"].groupby("target").r_direct_own_depth.mean()
    log(f"r_p table: {len(rp_real)} targets")

    pert_col_in_ceiling = next((c for c in ceiling.columns
                                if c.lower() in ("perturbation", "target", "target_gene", "pert")),
                               None)
    if pert_col_in_ceiling is None:
        raise KeyError(f"Could not find a perturbation-name column in ceiling results. "
                       f"Columns: {list(ceiling.columns)}")

    merged = ceiling.merge(rp_real.rename("r_p"), left_on=pert_col_in_ceiling, right_index=True)
    log(f"merged {len(merged)} perturbations (of {len(ceiling)} ceiling rows, "
        f"{len(rp_real)} r_p rows)")

    metric_cols = [c for c in merged.columns
                  if any(k in c.lower() for k in ("des", "pds", "mae")) and merged[c].dtype != object]
    corrs = {c: float(merged["r_p"].corr(merged[c])) for c in metric_cols}
    log(f"corr(r_p, oracle per-perturbation metric):\n{json.dumps(corrs, indent=2)}")

    merged.to_csv(os.path.join(OUT, "v5g1_rp_vs_oracle_difficulty_v1.csv"), index=False)
    with open(os.path.join(OUT, "v5g1_rp_vs_oracle_difficulty_report_v1.json"), "w") as f:
        json.dump({"n_matched": len(merged), "correlations": corrs}, f, indent=2)
    log("link stage complete.")


STAGES = dict(diagnose=stage_diagnose, baseline=stage_baseline,
             oracle=stage_oracle, score=stage_score, link=stage_link)

if __name__ == "__main__":
    # Kaggle runs this via papermill/IPython (colab_kernel_launcher.py), which
    # injects its own CLI args (-f /tmp/....json --HistoryManager...). A plain
    # argparse.parse_args() chokes on those with "unrecognized arguments" and
    # exits before stage 1 -- exactly what happened on the first Kaggle run.
    # parse_known_args() ignores anything it doesn't recognize instead.
    #
    # Stage selection, in priority order:
    #   1. --stage flag, if this is actually invoked from a real shell (local
    #      testing, or `!python ... --stage x` from a Kaggle code cell).
    #   2. V5G0_STAGE env var -- the reliable way to select a stage on Kaggle;
    #      set it in a cell with `import os; os.environ["V5G0_STAGE"]="diagnose"`
    #      before running this script, or via Kaggle's notebook environment
    #      variables panel.
    #   3. Default is "diagnose", NOT "all" -- a plain Run All on Kaggle must
    #      not silently trigger the expensive oracle pass before diagnose has
    #      confirmed the cell_eval API assumptions.
    ensure_deps()
    p = argparse.ArgumentParser()
    p.add_argument("--stage", default=None, choices=list(STAGES) + ["all"])
    args, _unknown = p.parse_known_args()
    stage = args.stage or os.environ.get("V5G0_STAGE", "diagnose")
    if stage not in list(STAGES) + ["all"]:
        raise SystemExit(f"Unknown stage '{stage}'. Choose from {list(STAGES)+['all']}.")
    log(f"stage selection: --stage={args.stage!r}  V5G0_STAGE env={os.environ.get('V5G0_STAGE')!r}  "
        f"-> running '{stage}'")
    order = ["diagnose", "baseline", "oracle", "score", "link"] if stage == "all" else [stage]
    for s in order:
        log(f"=== STAGE: {s} ===")
        STAGES[s]()